# 04 · Değerlendirme Metrikleri ve LLM-as-a-Judge (Aşama 4)

**Araştırma sorusu (RQ4).** GP + tabakalı seçimle eğitilen modeller, (a) IFD baseline'ına, (b) kümesiz ablasyona ve (c) tam veri SFT'sine karşı
**uzunluktan arındırılmış** olarak daha üstün mü? Hizalama (GRPO/DPO/R-DPO) bu üstünlüğü koruyor mu?

## 4.1 Metrikler

| Metrik | Tanım |
|---|---|
| Ortalama yanıt uzunluğu | $D_{test\text{-}250}$ istemlerinde açgözlü (greedy) üretimin simge sayısı (maks. 256) |
| Simge bazlı $D_{KL}$ | $\dfrac{1}{\sum_i\lvert\hat y_i\rvert}\sum_i\sum_t\sum_v\pi_\theta(v\mid\cdot)\log\dfrac{\pi_\theta(v\mid\cdot)}{\pi_{ref}(v\mid\cdot)}$, $\hat y_i\sim\pi_\theta$ (kendi çıktısı). Hizalı modellerde $\pi_{ref}$ = kaynak SFT modeli, SFT modellerinde LoRA'sız taban |
| Kazanma oranı | gpt-4o ikili hakem; **iki sıra** $(A,B)$ ve $(B,A)$ → konum yanlılığı giderilir. Skor: kazan = 1, beraberlik = ½, kayıp = 0 |
| **LC kazanma oranı** | $P(\text{kazan})=\sigma\big(b+w\tanh(\Delta L/s_{\Delta L})\big)$ lojistik regresyonu (AlpacaEval-LC ilkesi); rapor edilen değer $\sigma(b)$, yani **uzunluk farkı sıfırken** beklenen kazanma olasılığı. $\Delta L$ = model − rakip simge sayısı farkı |
| Belirsizlik | İstem düzeyinde **bootstrap** (300 yeniden örnekleme) ile %95 güven aralığı; istem düzeyinde **işaret testi** (iki-yönlü, ikili) |
| Konum yanlılığı | Hakemin "A" ve "B" konumlarını seçme oranları (≈ %50/%50 beklenir) |
| Sıra tutarlılığı | Aynı çift için iki sıralamada aynı sonucun verildiği istem oranı |

Hakem **yanıtların hangi modele ait olduğunu görmez** (körleştirme); yalnızca A/B etiketleri ve bağlam sunulur. `temperature=0`, JSON çıktı biçimi zorunlu.

In [ ]:
# --- Ortam kurulumu (Colab + yerel) ---------------------------------------------------
import os, sys, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

# Colab uzantısında çekirdek uzak makinede çalışır; yerel src/ orada olmadığından depo klonlanır.
if "google.colab" in sys.modules:
    PROJECT = Path("/content/gp-stratified-sft-selection")
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/suleymangzy/gp-stratified-sft-selection.git", str(PROJECT)], check=True)
    else:
        subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=False)
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["PROJECT_ROOT"] = str(PROJECT)                  # çıktılar PROJECT/outputs altına yazılır
sys.path.insert(0, str(PROJECT / "src"))

%matplotlib inline
import numpy as np, pandas as pd
from IPython.display import display, Markdown
from analysis import set_academic_style, show_figure, show_table
set_academic_style()
print("Proje dizini:", PROJECT)

## 4.2 Yapılandırma

In [ ]:
SMOKE = False              # True: 8 istem, 32 yeni simge (yalnızca kod yolu denetimi)
MAX_NEW_TOKENS = 256
JUDGE_N = 0                # 0 = tüm 250 test istemi
PAIRS = "key"              # "key": baseline + önerilen çiftler | "all": tüm model çiftleri
kw = dict(limit=8, max_new_tokens=32, judge_n=8, n_boot=50) if SMOKE else dict(max_new_tokens=MAX_NEW_TOKENS, judge_n=JUDGE_N)

from common import MODEL_DIR, ALL_MODELS
from evaluation import run_evaluation
ready = [m for m in ALL_MODELS if (MODEL_DIR / m / "adapter_config.json").exists()]
print(f"Hazır adaptör: {len(ready)}/{len(ALL_MODELS)} ->", ready)

## 4.3 Üretim, uzunluk ve $D_{KL}$
Üretim, 4-bit 3B tabanın üzerine adaptörleri sırayla takarak yapılır; yanıtlar `outputs/generations/*.jsonl` olarak saklanır (hakem aşaması bunları okur).

In [ ]:
out = run_evaluation(stage="generate", **kw)

In [ ]:
from analysis import eval_table, plot_length_kl
et = eval_table()
show_table(et, "tab_eval_metrics", "Test istemlerinde ortalama yanıt uzunluğu ve simge bazlı $D_{KL}$.", floatfmt="{:.3f}")
if len(et):
    show_figure(plot_length_kl(et), "fig_length_kl", "Ortalama yanıt uzunluğu (sol) ve $D_{KL}$ (sağ). KL referansı model türüne göre farklıdır (tabloya bakınız).")

## 4.4 LLM-as-a-Judge (gpt-4o)
Önce **kuru çalıştırma** ile çağrı sayısı gösterilir; maliyet yaklaşık $\text{çift sayısı}\times250\times2$ çağrıdır.

In [ ]:
import os, getpass
dry = run_evaluation(stage="judge", dry_run=True, pairs=PAIRS, **kw)["judge"]
print(dry)
if "OPENAI_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

In [ ]:
RUN_JUDGE = True
if RUN_JUDGE:
    judge = run_evaluation(stage="judge", pairs=PAIRS, **kw)["judge"]

## 4.5 Bulgular

In [ ]:
from analysis import pairwise_table, plot_forest, plot_lc_heatmap, position_bias_table
pw = judge["pairwise"]
show_table(pairwise_table(pw), "tab_pairwise",
           "gpt-4o ikili karşılaştırmaları: ham ve uzunluktan arındırılmış (LC) kazanma oranı, istem düzeyinde bootstrap %95 GA, Kazanç/Beraberlik/Yenilgi sayıları (istem düzeyi) ve işaret testi p-değeri.",
           floatfmt="{:.3f}")
show_figure(plot_forest(pw), "fig_forest", "LC kazanma oranları ve %95 güven aralıkları (kesikli çizgi = eşitlik, 50 %). Aralığı 50'yi içermeyen karşılaştırmalar anlamlıdır.")
show_figure(plot_lc_heatmap(judge["lc_matrix"]), "fig_lc_matrix", "Length-Controlled Win-Rate % matrisi (satır modeli sütun rakibine karşı; ölçülmeyen hücreler boş).")
show_table(position_bias_table(judge["position_bias"]), "tab_position_bias", "Hakemin konum yanlılığı: A/B konumu seçim oranları.", floatfmt="{:.1f}")

### Çoklu karşılaştırma uyarısı
Tablo birden çok eşleşme içerir; işaret testi $p$-değerleri **düzeltilmemiştir**. Anlamlılık iddiasında bulunulacaksa Bonferroni
($\alpha/m$) veya Holm düzeltmesi uygulayın. Aşağıdaki hücre Holm düzeltmesini hesaplar.

In [ ]:
p = pw["sign_test_p"].to_numpy()
order = np.argsort(p); m = len(p)
holm = np.empty(m); run = 0.0
for rank, i in enumerate(order):
    run = max(run, min(1.0, (m - rank) * p[i])); holm[i] = run
adj = pd.DataFrame({"model": pw["model"], "rakip": pw["opponent"], "p (ham)": p, "p (Holm)": holm,
                    "anlamlı (α=0.05)": holm < 0.05}).set_index(["model", "rakip"])
show_table(adj, "tab_holm", "İşaret testi p-değerleri ve Holm düzeltmesi (m = karşılaştırma sayısı).", floatfmt="{:.4f}")

## 4.6 Tüm sonuçların tek özeti

In [ ]:
from analysis import cost_table
summary = et.join(cost_table()[["süre (dk)", "tepe VRAM (GB)"]], how="left")
show_table(summary, "tab_final_summary", "Tüm modeller için uzunluk, $D_{KL}$ ve eğitim maliyeti özeti.", floatfmt="{:.2f}")

## 4.7 Tartışma *(deney sonrası doldurulacak)*
1. **Ana iddia:** `Model_SFT_GP_Strat1k` – `Model_SFT_IFD1k` karşılaştırmasında LC kazanma oranı ve %95 GA nedir; Holm sonrası anlamlı mı?
2. **Ablasyon:** Kümeleme (tabakalama) katkısı = `GP_Strat1k` vs `GP_Top1k`; **tam veri** `SFT_Full`'a göre 1.000 örnek yeterli mi?
3. **Hizalama:** GP başlangıcının üstünlüğü GRPO/DPO/R-DPO sonrası korunuyor mu? R-DPO, DPO'ya göre yanıt uzunluğunu ve LC oranını nasıl değiştirdi?
4. **Sapma–kazanç dengesi:** $D_{KL}$ ile LC kazanma oranı arasında bir takas var mı?
5. **Konum yanlılığı ve tutarlılık:** A/B seçim oranları ve sıra tutarlılığı hakemin güvenilirliği hakkında ne söylüyor?

### Geçerlilik tehditleri (rapora aynen aktarılabilir)
* **İç geçerlilik:** tek tohum, tek koşu; 4-bit nicemleme ve LoRA'nın etkisi tam fine-tuning'den farklı olabilir.
* **Yapı geçerliliği:** gpt-4o hakemi tek bir model; insan değerlendirmesiyle (ör. 50 örneklik alt küme) çapraz doğrulama önerilir. LC düzeltmesi uzunluk yanlılığını azaltır, tamamen ortadan kaldırmaz.
* **Dış geçerlilik:** yalnızca Qwen2.5 ailesi, İngilizce talimat verisi; `no_robots` test örnekleri 250 ile sınırlıdır (GA genişliğine bakın).
* **Veri sızıntısı önlemi:** GP, $D_{val\text{-}250}$ üzerinde optimize edildi; tüm nihai sonuçlar ayrık $D_{test\text{-}250}$ üzerindedir (defter 00).
* **Maliyet:** yargı çağrıları önbelleklenir (`outputs/judge/judgements.jsonl`), aynı çiftler yeniden ücretlendirilmez.

## 4.8 Rapor paketi
Aşağıdaki hücre tüm tablo (CSV/LaTeX) ve şekilleri (PNG/PDF) tek bir arşivde toplar.

In [ ]:
import shutil
from common import OUT
arch = shutil.make_archive(str(OUT / "rapor_paketi"), "zip", root_dir=OUT, base_dir="tables")
shutil.make_archive(str(OUT / "rapor_sekilleri"), "zip", root_dir=OUT, base_dir="figures")
print("Oluşturuldu:", arch, "ve", OUT / "rapor_sekilleri.zip")

# Colab VM'de outputs/ oturum kapanınca silinir: tüm sonuçları yerel makineye indirin.
from common import export_outputs
export_outputs("outputs_04")